# IVF Insight — Phase 5.1: Statistical Integrity & Interpretation Audit

**Author:** Senior Biomedical Data Scientist & Research Engineer  
**Dataset:** `data/raw/Dataset Clean.xlsx` (5,000 candidate cycles; rows 0–4999 retained)  
**Audit Focus:** Deconstruct apparent paradoxes (Female Age positive beta), unmask severe multicollinearity, audit calibration equations (CITL vs two-parameter slope), clarify variable count semantics, and eliminate untestable causal speculation.  

> [!IMPORTANT]
> **Clinical Non-Claim & Scientific Governance Notice:** This project is an educational data science portfolio. It does **not** constitute clinical decision support or treatment recommendations. Predictive importance must **never** be interpreted as causal efficacy.

## 1. Setup & Audit Module Imports

Initialize dependencies and import analytical functions from `ivf_insight.integrity_audit`.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import Image, display

from ivf_insight.eda import load_clean_cohorts
from ivf_insight.integrity_audit import (
    audit_temporal_drift_hypotheses,
    audit_female_age_association,
    audit_logistic_regression_diagnostics,
    audit_calibration_equations,
    audit_feature_definitions,
)

REPORTS_DIR = Path('../reports')
FIG_DIR = REPORTS_DIR / 'robustness_figures'
print('Loaded integrity audit module. Reports directory:', REPORTS_DIR.resolve())


## 2. Temporal Drift Interpretation Audit

Distinguishing between observed data facts (reduction in transferred embryos, younger male age) and untestable clinical speculations (blastocyst culture adoption, culture media changes).

In [ ]:
df_hyp = pd.read_csv(REPORTS_DIR / 'temporal_drift_hypotheses_audit.csv')
display(df_hyp)


## 3. Female Age Odds-Ratio Investigation: Suppression & Confounding

Deconstructing why Female Age has a positive standardized coefficient ($\beta = +0.3062$, $\text{OR} = 1.358$, $p < 0.0001$) in multivariable regression. We evaluate unadjusted vs adjusted models, reciprocal suppression with Male Age, and empirical age bracket outcomes.

In [ ]:
df_age_prog = pd.read_csv(REPORTS_DIR / 'age_association_analysis.csv')
display(df_age_prog)

df_brackets = pd.read_csv(REPORTS_DIR / 'age_bracket_prevalence.csv')
display(df_brackets)

if (FIG_DIR / 'age_association_splines.png').exists():
    display(Image(filename=str(FIG_DIR / 'age_association_splines.png'), width=900))


## 4. Logistic Regression Diagnostics: Multicollinearity & Influence

Auditing Variance Inflation Factors (VIF), Box-Tidwell linearity in logit, condition numbers, and bootstrap confidence intervals ($B=500$) for the Pre-Transfer Scenario C model.

In [ ]:
df_lr_diag = pd.read_csv(REPORTS_DIR / 'logistic_regression_diagnostics.csv')
display(df_lr_diag[['Feature', 'Standardized_Beta_Wald', 'Wald_Odds_Ratio', 'Wald_OR_95_CI', 'Bootstrap_OR_95_CI', 'Variance_Inflation_Factor', 'Collinearity_Flag', 'Logit_Linearity_Status']])

if (FIG_DIR / 'multicollinearity_vif.png').exists():
    display(Image(filename=str(FIG_DIR / 'multicollinearity_vif.png'), width=800))


## 5. Calibration Audit: Canonical CITL vs Recalibration Slope

Auditing Calibration-in-the-Large (CITL via offset) versus unconstrained two-parameter recalibration intercept and slope.

In [ ]:
df_calib_audit = pd.read_csv(REPORTS_DIR / 'calibration_audit.csv')
display(df_calib_audit[['Scenario', 'Model', 'Observed_Holdout_Prevalence', 'Mean_Predicted_Probability', 'Expected_to_Observed_Ratio_EO', 'CITL_Offset_Alpha', 'CITL_95_CI', 'Calibration_Slope_Beta', 'Calibration_Slope_95_CI', 'Calibration_Interpretation']])

if (FIG_DIR / 'calibration_citl_recalibration.png').exists():
    display(Image(filename=str(FIG_DIR / 'calibration_citl_recalibration.png'), width=750))


## 6. Feature Semantics & Explainability Audit

Clarifying integer count definitions for `frezzing` (count of cryopreserved surplus embryos, not binary), `Number of embryos`, and `Embryo Grade 1..4`, while strictly separating odds ratios from Random Forest Gini importances.

In [ ]:
df_feats = pd.read_csv(REPORTS_DIR / 'feature_interpretation_audit.csv')
display(df_feats)


## 7. Decision Threshold Interpretation Safeguards

1. **Exploratory Status:** Decision threshold curves are exploratory statistical sensitivity analyses, not clinical policy.
2. **Prevalence $\ne$ Valid Cutoff:** The empirical holdout prevalence ($0.787$) is not a clinically validated decision threshold.
3. **Bayes' Theorem Base Rate:** High positive prevalence ($78.7\%$) inflates PPV to $86.8\%$ while depressing NPV to $41.6\%$ at threshold $0.5$.
4. **Clinical Governance:** No cutoff is recommended for treatment or patient selection without an explicit clinical utility/loss matrix.
